In [ ]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7 googletrans==4.0.0-rc1 triton==2.0.0 bitsandbytes==0.40.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [ ]:
import json
from googletrans import Translator
import random
from tqdm import tqdm
import time
# import os
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, HfArgumentParser, pipeline, logging
# from peft import LoraConfig, PeftModel
# from trl import SFTTrainer

In [ ]:
class MealPlanDataset():
    def __init__(self):
        self.translator = Translator()
        self.translation_dict = {
            "Breakfast": "Bữa sáng",
            "Lunch": "Bữa trưa",
            "Dinner": "Bữa tối",
            "serving": "phần",
            "and": "và",
            "with": "với",
            "&": "và",
            ",": ",",
        }
        self.ingredients = {
            "chicken": "thịt gà",
            "beef": "thịt bò",
            "pork": "thịt heo",
            "salmon": "cá hồi",
            "shrimp": "tôm",
            "crab": "cua",
            "vegetable": "rau",
            "veggie": "rau củ",
            "yogurt": "sữa chua",
            "milk": "sữa",
            "cheese": "phô mai",
            "egg": "trứng",
            "tofu": "đậu phụ",
            "mushroom": "nấm",
            "potato": "khoai tây",
            "rice": "gạo",
            "noodle": "mì",
            "bread": "bánh mì",
            "fish": "cá",
            "pasta": "mì ống",
            "spinach": "rau chân vịt",
            "tomato": "cà chua",
            "carrot": "cà rốt",
            "broccoli": "bông cải xanh",
            "apple": "táo",
            "banana": "chuối",
            "orange": "cam",
            "strawberry": "dâu tây",
            "blueberry": "việt quất",
            "avocado": "bơ",
        }

    def clean_text(self, text):
        """Làm sạch văn bản trước khi dịch"""
        return text.strip().lower()

    def translate_text(self, text):
        """Dịch văn bản với cache và xử lý lỗi"""
        text = self.clean_text(text)
        if text in self.translation_dict:
            return self.translation_dict[text]
        time.sleep(0.5)

        try:
            translated = self.translator.translate(text, src="en", dest="vi").text
            return translated.lower().strip()
        except Exception as e:
            print(f"Lỗi khi dịch '{text}': {e}")
            return text

    def extract_and_translate_foods(self, file_path):
        """Lọc và dịch các cụm món ăn từ dữ liệu JSON"""
        food_set = set()

        # Duyệt qua tất cả các tệp JSON trong thư mục
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Duyệt qua từng mục trong dữ liệu
        for calorie_range, meal_plans in data.items():
            for plan in meal_plans:
                for meal_type, items in plan.items():
                    for item in items:
                        food_set.add(item["food"])

        # Dịch các cụm món ăn và thêm vào translation_dict
        for food in sorted(food_set):
            translated_food = self.translate_text(food)
            self.translation_dict[food] = translated_food
            print(f"{food} -> {translated_food}")

    def analyze_ingredients(self, food_name):
        """Phân tích thành phần từ tên món ăn"""
        detected = []
        separators = [" and ", " with ", " & ", ", "]  # Các dấu phân cách phổ biến
        for sep in separators:
            if sep in food_name:
                parts = food_name.split(sep)
                for part in parts:
                    part = part.strip().lower()
                    for eng, vi in self.ingredients.items():
                        if eng in part:
                            detected.append(vi)
                return detected if detected else ["đa dạng"]

        # Nếu không phải từ ghép, kiểm tra toàn bộ
        food_lower = food_name.lower()
        for eng, vi in self.ingredients.items():
            if eng in food_lower:
                detected.append(vi)
        return detected if detected else ["đa dạng"]

    def generate_prompt_from_meal(self, calories, meal_plan):
        """Tạo prompt dựa trên thực đơn cụ thể"""
        all_ingredients = []
        for items in meal_plan.values():
            for item in items:
                all_ingredients.extend(self.analyze_ingredients(item["food"].lower()))

        main_ingredients = random.sample(
            list(set(all_ingredients)), min(2, len(set(all_ingredients)))
        )

        templates = [
            "Tôi muốn thực đơn khoảng {} calo với các món chứa {}.",
            "Gợi ý thực đơn {} calo có các món chính là {}.",
            "Hãy đề xuất thực đơn khoảng {} calo. Tôi thích ăn {}",
            "Thực đơn {} calo với thành phần chính là {}.",
            "Tôi cần thực đơn {} calo/ngày, ưu tiên món có {}.",
        ]

        prompt = random.choice(templates).format(
            calories, " và ".join(main_ingredients)
        )

        return prompt

    def process_data(self, input_file, output_file):
        """Xử lý dữ liệu từ file JSON và tạo hội thoại"""
        with open(input_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        transformed_data = []

        for calories, meal_plans in tqdm(data.items(), desc="Xử lý"):
            for plan in meal_plans:
                if not plan:  # Bỏ qua các phần tử rỗng
                    continue
                translated_plan = {}
                for meal_type, items in plan.items():
                    translated_plan[self.translate_text(meal_type)] = [
                    {
                        "food": self.translate_text(item["food"]).lower(),
                        "serving": self.translate_text(item["serving"]).lower(),
                    }
                    for item in items
                ]

                prompt = self.generate_prompt_from_meal(calories, plan)

                response = "Thực đơn gợi ý:\n"
                for meal_type, items in translated_plan.items():
                    response += f"- {meal_type}: "
                    response += ", ".join(
                        [f"{item['food']} ({item['serving']})" for item in items]
                    )
                    response += "\n"

                conversation = f"""<|im_start|>system
Bạn là chuyên gia dinh dưỡng AI<|im_end|>
<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
{response.strip()}<|im_end|>"""

                transformed_data.append({'text': ''.join(conversation)})
        return transformed_data

In [ ]:
class DataPreprocessor:
    def __init__(self):
        self.dataset = []

    def load_and_process(self, json_files):
        """Tải và xử lý tất cả file JSON"""
        for file in tqdm(json_files, desc="Processing files"):
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            processor = MealPlanDataset()
            processor.extract_and_translate_foods(file)
            processed = processor.process_data(file, None)  # Không lưu file tạm
            self.dataset.extend(processed)

        return self.dataset

    def save_to_jsonl(self, output_file):
        """Lưu dữ liệu đã xử lý dưới dạng JSONL"""
        with open(output_file, 'w', encoding='utf-8') as f:
            for item in self.dataset:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [ ]:
# json_files = ['/kaggle/input/raw-meal-datasets/meal_data_1500-1900.json',
#             '/kaggle/input/raw-meal-datasets/meal_data_1000-1400.json',
#               '/kaggle/input/raw-meal-datasets/meal_data_2000-2400.json',
#               '/kaggle/input/raw-meal-datasets/meal_data_2500-2900.json',
#               '/kaggle/input/raw-meal-datasets/meal_data_3000-3500.json']
json_files = [
            '/content/drive/MyDrive/meal_data_3500.json'
            ]
preprocessor = DataPreprocessor()
dataset = preprocessor.load_and_process(json_files)
print("Số dòng dữ liệu đã xử lý:", len(dataset))
print("Dòng đầu tiên:")
print(dataset[0] if dataset else "Không có dữ liệu.")
preprocessor.save_to_jsonl('/content/drive/MyDrive/pre_meal_data_3500.jsonl')

Processing files:   0%|          | 0/1 [00:00<?, ?it/s]

"Banana Cream Pie" Oatmeal -> bánh kem "kem chuối"
1000 Calorie Shake -> 1000 calo lắc
3 Bean Salad -> 3 món salad đậu
5-Minute Healthy Strawberry Frozen Yogurt -> sữa chua đông lạnh sữa đông lạnh 5 phút
All American Tuna -> tất cả cá ngừ mỹ
Almond Butter Banana Toast -> bánh mì nướng bơ hạnh nhân
Almonds -> hạnh nhân
Almonds and Blueberries Yogurt Snack -> hạnh nhân và quả việt quất sữa chua
Ants on a Log -> kiến trên một khúc gỗ
Apple -> quả táo
Apple Celery Salad -> salad cần tây táo
Apple Pie Kefir Smoothie -> smoothie bánh kefir táo
Apple Slice Sandwich -> bánh sandwich lát táo
Apple Toast -> bánh mì nướng táo
Apple Walnut Parfait -> quả óc chó táo parfait
Apples and Almond Butter -> táo và bơ hạnh nhân
Apples and Almond Butter on Toast -> táo và bơ hạnh nhân trên bánh mì nướng
Arugula Pine Nut Salad -> salad hạt thông arugula
Arugula Salad with Grapefruit dressing -> salad arugula với nước sốt bưởi
Arugula and Avocado Salad With Shaved Parmesan -> arugula và salad bơ với parmesan


Processing files: 100%|██████████| 1/1 [2:39:31<00:00, 9571.48s/it]

Số dòng dữ liệu đã xử lý: 1000
Dòng đầu tiên:
{'text': '<|im_start|>system\nBạn là chuyên gia dinh dưỡng AI<|im_end|>\n<|im_start|>user\nTôi muốn thực đơn khoảng 3500 calo với các món chứa cá hồi và dâu tây.<|im_end|>\n<|im_start|>assistant\nThực đơn gợi ý:\n- bữa sáng: chuối bơ đậu phộng chia pudding (1 phục vụ), dâu tây kiwi bơ đậu phộng sinh tố ăn sáng (1 phục vụ)\n- bữa trưa: caprese corn bữa trưa salad (2 phần ăn), dâu tây (1 cốc)\n- bữa tối: salmon salmon cay california (4 ngăn xếp), pan fried corn (1 phục vụ)<|im_end|>'}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
